In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json

current_dir = Path.cwd()
project_root = current_dir.parents[2]
path_data = project_root / 'DATA_PPMI/Results/MODEL_DATA/DATA_BL_V08.csv'

data = pd.read_csv(path_data,index_col=0)
patnos = data.index.unique().to_list()

updrs3_cols=['NP3SPCH','NP3FACXP','NP3RIGN','NP3RIGRU','NP3RIGLU','NP3RIGRL','NP3RIGLL','NP3FTAPR','NP3FTAPL','NP3HMOVR','NP3HMOVL','NP3PRSPR','NP3PRSPL','NP3TTAPR','NP3TTAPL','NP3LGAGR','NP3LGAGL','NP3RISNG','NP3GAIT','NP3FRZGT','NP3PSTBL','NP3POSTR','NP3BRADY','NP3PTRMR','NP3PTRML','NP3KTRMR','NP3KTRML','NP3RTARU','NP3RTALU','NP3RTARL','NP3RTALL','NP3RTALJ','NP3RTCON']

def HY3_classification(nhy):
    if nhy == 0:
        return 0  # Healthy
    elif nhy  in [1,2]:
        return 1  # Early Stage
    else:
        return 2  # Advanced Stage

def HY4_classification(nhy):
    if nhy == 0:
        return 0  # Healthy
    elif nhy == 1:
        return 1  # Early Stage
    elif nhy == 2:
        return 2  # Moderate Stage
    else:
        return 3  # Advanced Stage


data["STAGE_NHY3"] = data["NHY"].apply(HY3_classification)
data["STAGE_NHY4"] = data["NHY"].apply(HY4_classification)

V08_cols = ['ENRLLRRK2', 'ENRLGBA', 'COHORT_DEFINITION', 'SEX', 'RAWHITE', 'EDUCYRS', 'AGE_AT_VISIT', "STAGE_NHY3", "STAGE_NHY4"]
data_V08 = data[V08_cols+[ 'EVENT_ID',"NHY"]+updrs3_cols].copy()

data_V08 = data_V08.loc[data_V08["EVENT_ID"] == 'V08',:]  # last visit
data_V08.drop(columns=["EVENT_ID"], inplace=True)
data_V08['UPDRS3_TOTAL_V08'] = data_V08[updrs3_cols].sum(axis=1)
data_V08.drop(columns=updrs3_cols, inplace=True)

data_V06 = data[V08_cols+[ 'EVENT_ID',"NHY"]+updrs3_cols].copy()
data_V06['UPDRS3_TOTAL_V06'] = data_V06[updrs3_cols].sum(axis=1)
data_V06.drop(columns=updrs3_cols, inplace=True)
data_V06 = data_V06.loc[data_V06["EVENT_ID"] == 'V06',:]  # actual visit
data_V06.drop(columns=["EVENT_ID"], inplace=True)

print("data_V08:", data_V08.shape)
print("data_V06:", data_V06.shape)

data_V08 = data_V08.merge(
    data_V06[['UPDRS3_TOTAL_V06']],
    left_index=True,
    right_index=True,
    how='inner')

print("data_V08 after merge:", data_V08.shape)
data_V08['Delta_UPDRS3'] = data_V08['UPDRS3_TOTAL_V08'] - data_V08['UPDRS3_TOTAL_V06']

def classification_mcid(x):
    if x <= -3:
        return 0  # Clinically meaningful improvement
    elif x >= 4:
        return 2   # Clinically meaningful worsening
    else:
        return 1   # Stable / no clinically meaningful change

data_V08['MCID_CLASS'] = data_V08['Delta_UPDRS3'].apply(classification_mcid)
data_V08.drop(columns=['UPDRS3_TOTAL_V06','COHORT_DEFINITION'], inplace=True)
data_V08.head(10)



data_V08: (909, 11)
data_V06: (909, 11)
data_V08 after merge: (909, 12)


,ENRLLRRK2,ENRLGBA,SEX,RAWHITE,EDUCYRS,AGE_AT_VISIT,STAGE_NHY3,STAGE_NHY4,NHY,UPDRS3_TOTAL_V08,Delta_UPDRS3,MCID_CLASS
PATNO,,,,,,,,,,,,
3003,0,0,0.0,1.0,16.0,59.7,1,2,2.0,46.0,3.0,1
3018,0,0,0.0,1.0,16.0,63.6,1,2,2.0,38.0,3.0,1
3020,0,0,0.0,1.0,15.0,77.0,2,3,3.0,45.0,9.0,2
3028,0,0,1.0,1.0,22.0,78.8,1,2,2.0,36.0,-13.0,0
3051,0,0,1.0,1.0,18.0,74.7,1,2,2.0,22.0,3.0,1
3054,0,0,1.0,1.0,16.0,77.3,1,2,2.0,24.0,1.0,1
3056,0,0,1.0,1.0,16.0,58.7,1,2,2.0,31.0,-2.0,1
3058,0,0,1.0,1.0,19.0,82.8,2,3,3.0,53.0,9.0,2
3060,0,0,1.0,1.0,16.0,78.1,1,1,1.0,12.0,-2.0,1


In [2]:
other_cols = [col for col in data.columns if col not in V08_cols]
data_removed = data[V08_cols].copy()
data_stats = data[other_cols].copy()
data_stats = data_stats.loc[data_stats["EVENT_ID"].isin(['BL','V04','V06']),:]  # last visit
data_stats.drop(columns=["EVENT_ID"], inplace=True)

df_grouped = data_stats.groupby(level="PATNO")

df_mean = df_grouped.mean().add_suffix("_mean")
df_min  = df_grouped.min().add_suffix("_min")
df_max  = df_grouped.max().add_suffix("_max")
df_var  = df_grouped.var().add_suffix("_var")
df_std  = df_grouped.std().add_suffix("_std")

data_stats = pd.concat([df_mean, df_min, df_max, df_var, df_std], axis=1)
final_data = data_V08.merge(data_stats, left_index=True, right_index=True, how='inner')
print("final_data:", final_data.shape)

final_data: (909, 942)


# Cols x Domain

In [3]:
cols_dict ={}

## SC

In [4]:
SC_data = pd.read_csv(project_root / 'DATA_PPMI/Results/SUBJECT_CHARACTERISTICS.csv')
SC_data_cols = SC_data.columns.tolist()
SC_data_cols.remove('PATNO')
SC_data_cols.remove('EVENT_ID')
SC_data_cols.remove('COHORT_DEFINITION')
cols_dict['SC_data'] = SC_data_cols

## Motor

In [5]:
M_data = pd.read_csv(project_root / 'DATA_PPMI/Results/MDS_UPDR.csv')

M_data_cols = M_data.columns.tolist()
M_data_cols.remove('PATNO')
M_data_cols.remove('EVENT_ID')
M_data_cols.remove('NHY')

stats_cols_M = []

for col in M_data_cols:
    for suffix in ["_mean", "_min", "_max", "_var", "_std"]:
        stats_cols_M.append(col + suffix)


cols_dict['M_data_STATS'] = stats_cols_M
cols_dict['M_data_V06'] = [col + "_V06" for col in M_data_cols]
cols_dict['M_data_DELTA'] = [c + "_delta_V06_V04" for c in M_data_cols] + [c + "_delta_V06_BL" for c in M_data_cols] + [c + "_delta_V04_BL" for c in M_data_cols]


dict_updrs3 = {}
dict_updrs3['SC_data'] = SC_data_cols
dict_updrs3['UPDRS_NO_MOTOR_STATS'] = [col for col in stats_cols_M if col.startswith('NP1')]
dict_updrs3['UPDRS_IMPACTO_MOTOR_STATS'] = [col for col in stats_cols_M if col.startswith('NP2')]
dict_updrs3['UPDRS_MOTOR_STATS'] = [col for col in stats_cols_M if col.startswith('NP3')]

dict_updrs3['UPDRS_NO_MOTOR_V06'] = [col for col in M_data_cols if col.startswith('NP1')]
dict_updrs3['UPDRS_NO_MOTOR_V06'] = [c + "_V06" for c in dict_updrs3['UPDRS_NO_MOTOR_V06']]

dict_updrs3['UPDRS_IMPACTO_MOTOR_V06'] = [col for col in M_data_cols if col.startswith('NP2')]
dict_updrs3['UPDRS_IMPACTO_MOTOR_V06'] = [c + "_V06" for c in dict_updrs3['UPDRS_IMPACTO_MOTOR_V06']]

dict_updrs3['UPDRS_MOTOR_V06'] = [col for col in M_data_cols if col.startswith('NP3')]
dict_updrs3['UPDRS_MOTOR_V06'] = [c + "_V06" for c in dict_updrs3['UPDRS_MOTOR_V06']]

dict_updrs3['UPDRS_NO_MOTOR_delta'] = [col for col in M_data_cols if col.startswith('NP1')]
dict_updrs3['UPDRS_NO_MOTOR_delta'] = [c + "_delta_V06_V04" for c in dict_updrs3['UPDRS_NO_MOTOR_delta']] + [c + "_delta_V06_BL" for c in dict_updrs3['UPDRS_NO_MOTOR_delta']] + [c + "_delta_V04_BL" for c in dict_updrs3['UPDRS_NO_MOTOR_delta']]
dict_updrs3['UPDRS_IMPACTO_MOTOR_delta'] = [col for col in M_data_cols if col.startswith('NP2')]
dict_updrs3['UPDRS_IMPACTO_MOTOR_delta'] = [c + "_delta_V06_V04" for c in dict_updrs3['UPDRS_IMPACTO_MOTOR_delta']] + [c + "_delta_V06_BL" for c in dict_updrs3['UPDRS_IMPACTO_MOTOR_delta']] + [c + "_delta_V04_BL" for c in dict_updrs3['UPDRS_IMPACTO_MOTOR_delta']]
dict_updrs3['UPDRS_MOTOR_delta'] = [col for col in M_data_cols if col.startswith('NP3')]
dict_updrs3['UPDRS_MOTOR_delta'] = [c + "_delta_V06_V04" for c in dict_updrs3['UPDRS_MOTOR_delta']] + [c + "_delta_V06_BL" for c in dict_updrs3['UPDRS_MOTOR_delta']] + [c + "_delta_V04_BL" for c in dict_updrs3['UPDRS_MOTOR_delta']]

with open(project_root /"SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/UPDRS_Domain_data.json", "w", encoding="utf-8") as archivo:
    json.dump(dict_updrs3, archivo, indent=4, ensure_ascii=False)


## Non-Motor

In [6]:
NM_data = pd.read_csv(project_root/ 'DATA_PPMI/Results/NON_MOTOR_DATA.csv' )
NM_data_cols = NM_data.columns.tolist()
NM_data_cols.remove('PATNO')
NM_data_cols.remove('EVENT_ID')
stats_cols_NM = []
for col in NM_data_cols:
    for suffix in ["_mean", "_min", "_max", "_var", "_std"]:
        stats_cols_NM.append(col + suffix)

cols_dict['NM_data_STATS'] = stats_cols_NM
cols_dict['NM_data_V06'] = [col + "_V06" for col in NM_data_cols]
cols_dict['NM_data_DELTA'] = [c + "_delta_V06_V04" for c in NM_data_cols] + [c + "_delta_V06_BL" for c in NM_data_cols] + [c + "_delta_V04_BL" for c in NM_data_cols]


# DaTScan, uso en Standby reduccion significativa de datos

In [7]:
data_img = pd.read_csv(project_root / 'DATA_PPMI/Results/DatScan_SBR.csv')
data_img = data_img.loc[data_img["PATNO"].isin(patnos) & data_img["EVENT_ID"].isin(['BL', 'V04', 'V06', 'V08']),:]  # last visit
data_img['New_ADQUISITION'] = data_img['EVENT_ID'].map({'BL': 0, 'V04': 1, 'V06': 2, 'V08': 3})
data_img['PATNO'].nunique() # reduccion de pacientes elevada
data_img = data_img.loc[data_img.groupby('PATNO')['New_ADQUISITION'].idxmax()]
data_img.drop(columns=['EVENT_ID', 'New_ADQUISITION'], inplace=True)
data_img.set_index('PATNO', inplace=True)

left_cols = [c for c in data_img.columns if "_L_" in c]

for col_L in left_cols:
    
    col_R = col_L.replace("_L_", "_R_")
    
    if col_R in data_img.columns:
        base = col_L.replace("_L_", "_")
        
        data_img[base + "_MEAN_SIDES"] = data_img[[col_L, col_R]].mean(axis=1)
        data_img[base + "_STD_SIDES"] = data_img[[col_L, col_R]].std(axis=1)

# eliminar columnas izquierda y derecha
data_img.drop(columns=[c for c in data_img.columns if "_L_" in c or "_R_" in c], inplace=True)

cols_dict['DAT_data'] = data_img.columns.tolist()
data_img.to_csv(project_root / 'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/IMAGING/X_DAT_data.csv')



# JSON file 

In [8]:
with open(project_root /"SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/Domain_data.json", "w") as archivo:
    json.dump(cols_dict, archivo, indent=4)